# Nvidia NemoRetriever + FAISS 索引示例

本notebook演示如何使用FAISS向量索引加速大规模文档检索

## 1. 环境准备和导入

In [ ]:
%load_ext autoreload
%autoreload 2

# 加载 autoreload 扩展
# 设置自动重新加载模式。2 表示任何已导入的模块，只要其源代码发生变化，就会在执行下一行代码时自动重新加载

In [1]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

import torch
from transformers import AutoModel
from pdf2image import convert_from_path
from nvidia_rag_with_faiss import NvidiaRAGPipeline, GPUMemoryMonitor, ImageReranker

# 设置GPU设备
DEVICE = 9
torch.cuda.set_device(DEVICE)

print("✓ 环境准备完成")

/home/zechuan/miniconda3/envs/finance_RAG/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zechuan/miniconda3/envs/finance_RAG/lib/python3.11/site-packages/transformers/utils/hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


✓ 环境准备完成


## 2. 加载Retriever模型

In [2]:
memory_monitor = GPUMemoryMonitor(DEVICE)
memory_monitor.print_memory("Before loading model")

# 加载Nvidia NemoRetriever模型
retriever_model = AutoModel.from_pretrained(
    'nvidia/llama-nemoretriever-colembed-3b-v1',
    device_map=f'cuda:{DEVICE}',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    revision='50c36f4d5271c6851aa08bd26d69f6e7ca8b870c',
).eval()

memory_monitor.print_memory("After loading model")
print("✓ Retriever模型加载完成")

[Before loading model] GPU Memory - Allocated: 0.00GB, Reserved: 0.00GB


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.13it/s]

[After loading model] GPU Memory - Allocated: 8.21GB, Reserved: 8.34GB
✓ Retriever模型加载完成


## 3. 加载PDF文档

In [3]:
# 加载PDF
pdf_path = 'contents/2024_Tencent_ESG.pdf'
images = convert_from_path(pdf_path, dpi=200)

print(f"✓ PDF加载完成，共 {len(images)} 页")

✓ PDF加载完成，共 111 页


## 4. 初始化RAG Pipeline（带FAISS索引）

In [4]:
import numpy as np
# FAISS配置
faiss_config = {
    "embedding_dim": 3072,  # Nvidia NemoRetriever的嵌入维度
    "index_type": "ivfflat",  # 使用IVF索引，适合中等规模
    "nlist": int(np.sqrt(len(images))),  # 聚类中心数量（建议为sqrt(n_pages)）
    "use_gpu": True,  # FAISS GPU索引（可选）
}

# faiss_config = {
#     "index_type": "flat",  
#     "use_gpu": True,  # FAISS GPU索引（可选）
# }
# 初始化Pipeline
rag_pipeline = NvidiaRAGPipeline(
    retriever_model=retriever_model,
    use_faiss=True,
    faiss_config=faiss_config,
    device=DEVICE
)

print("✓ RAG Pipeline初始化完成")

✓ RAG Pipeline初始化完成


## 5. 编码文档并构建FAISS索引

In [5]:
# 编码文档
passage_embeddings, page_ids = rag_pipeline.encode_passages(
    images=images,
    batch_size=8,
    page_id_prefix="tencent_esg"
)

print(f"✓ 文档编码完成: {passage_embeddings.shape}")


编码 111 个文档页面...
[Before passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 8.34GB


Extracting document embeddings...: 100%|██████████| 14/14 [00:40<00:00,  2.86s/it]


[After passage encoding] GPU Memory - Allocated: 8.21GB, Reserved: 18.12GB
✓ 文档编码完成
✓ 文档编码完成: torch.Size([111, 1802, 3072])


In [6]:
# 构建FAISS索引
# rag_pipeline.build_index(save_dir="./faiss_index/tencent_esg")
# print("✓ FAISS索引构建完成")

rag_pipeline.load_index("./faiss_index/tencent_esg")
print("✓ FAISS索引加载完成")

✓ 索引已从 faiss_index/tencent_esg 加载
  - 索引类型: ivfflat
  - 向量数量: 200022
  - 页面数量: 111
✓ FAISS索引加载完成


## 6. 使用FAISS索引进行检索

In [7]:
# 定义查询
queries = [
    '2022年员工总数是多少？',
    '以2021年作为基准年，2023年温室气体范围3排放量是否达到了减少的目标？',
    '截至二零二四年每收入单位的温室气体排放总量是多少？'
]

# 批量检索（统一使用 queries 参数）
import time

print("\n" + "="*80)
print("使用FAISS索引检索（批量查询）")
print("="*80)

start_time = time.time()

# 批量检索 - 传入查询列表
all_results = rag_pipeline.retrieve(
    queries=queries,  # 统一使用 queries 参数
    top_k=20,
    use_index=True,
    batch_size=8
)

elapsed_time = time.time() - start_time

# 显示结果 - all_results 是 List[List[Dict]]
for i, (query, results) in enumerate(zip(queries, all_results)):
    print(f"\n查询 {i+1}: {query}")
    print("-"*80)
    print(f"Top-15 结果:")
    for rank, result in enumerate(results[:15], 1):
        print(f"  {rank}. 第 {result['page_num']} 页，分数: {result['score']:.4f}")

print(f"\n总检索时间: {elapsed_time:.3f}秒 (平均每个查询 {elapsed_time/len(queries):.3f}秒)")


print("\n" + "="*80)
print("单个查询示例")
print("="*80)

# 单个查询 - 也传入列表格式
single_query_results = rag_pipeline.retrieve(
    queries=["2022年员工总数是多少？"],  # 单个查询也用列表
    top_k=20,
    use_index=True
)

# 返回格式: [[results]]，取第一个元素
results = single_query_results[0]
print(f"\n查询: 2022年员工总数是多少？")
print(f"Top-5 结果:")
for rank, result in enumerate(results, 1):
    print(f"  {rank}. 第 {result['page_num']} 页，分数: {result['score']:.4f}")


使用FAISS索引检索（批量查询）


Extracting query embeddings...: 100%|██████████| 1/1 [00:02<00:00,  2.32s/it]


query_embedding.shape: (32, 3072)
query_embedding.shape: (32, 3072)
query_embedding.shape: (32, 3072)

查询 1: 2022年员工总数是多少？
--------------------------------------------------------------------------------
Top-15 结果:
  1. 第 63 页，分数: 27.9014
  2. 第 55 页，分数: 27.7525
  3. 第 45 页，分数: 27.4122
  4. 第 101 页，分数: 27.3640
  5. 第 75 页，分数: 27.2996
  6. 第 35 页，分数: 27.2633
  7. 第 13 页，分数: 26.6834
  8. 第 91 页，分数: 26.6214
  9. 第 90 页，分数: 25.8386
  10. 第 81 页，分数: 25.8289
  11. 第 33 页，分数: 25.8169
  12. 第 38 页，分数: 25.1452
  13. 第 43 页，分数: 24.9250
  14. 第 44 页，分数: 24.5208
  15. 第 19 页，分数: 24.0442

查询 2: 以2021年作为基准年，2023年温室气体范围3排放量是否达到了减少的目标？
--------------------------------------------------------------------------------
Top-15 结果:
  1. 第 104 页，分数: 26.2978
  2. 第 18 页，分数: 26.2757
  3. 第 17 页，分数: 24.6862
  4. 第 100 页，分数: 22.3862
  5. 第 21 页，分数: 19.9749
  6. 第 97 页，分数: 19.4116
  7. 第 22 页，分数: 17.4507
  8. 第 16 页，分数: 17.0577
  9. 第 103 页，分数: 16.1761
  10. 第 102 页，分数: 16.1003
  11. 第 95 页，分数: 15.8708
  12. 第 6 

Extracting query embeddings...: 100%|██████████| 1/1 [00:02<00:00,  2.79s/it]

query_embedding.shape: (14, 3072)

查询: 2022年员工总数是多少？
Top-5 结果:
  1. 第 63 页，分数: 9.8818
  2. 第 71 页，分数: 9.8561
  3. 第 40 页，分数: 9.8551
  4. 第 55 页，分数: 9.7375
  5. 第 45 页，分数: 9.3955
  6. 第 101 页，分数: 9.3482
  7. 第 75 页，分数: 9.2763
  8. 第 35 页，分数: 9.2399
  9. 第 96 页，分数: 9.0785
  10. 第 13 页，分数: 8.6641
  11. 第 91 页，分数: 8.6270
  12. 第 77 页，分数: 8.4282
  13. 第 29 页，分数: 7.9736
  14. 第 38 页，分数: 7.9230
  15. 第 90 页，分数: 7.8216
  16. 第 81 页，分数: 7.8129
  17. 第 33 页，分数: 7.8019
  18. 第 82 页，分数: 7.7703
  19. 第 43 页，分数: 6.9097
  20. 第 76 页，分数: 6.8416


## 7. 性能对比：FAISS vs 直接计算

In [8]:
import time

test_query = queries[0]
# test_queries = queries
# 方法1: 使用FAISS索引
start = time.time()
results_faiss = rag_pipeline.retrieve(test_query, top_k=20, use_index=True)
time_faiss = time.time() - start

# 方法2: 直接计算（无索引）
start = time.time()
results_direct = rag_pipeline.retrieve(test_query, top_k=20, use_index=False)
time_direct = time.time() - start

print("\n" + "="*80)
print("性能对比")
print("="*80)
print(f"FAISS索引检索: {time_faiss:.3f}秒")
print(f"直接计算检索: {time_direct:.3f}秒")
print(f"加速比: {time_direct/time_faiss:.2f}x")
print("\n注: 对于更大规模的文档集合，FAISS的优势会更明显")

Extracting query embeddings...: 100%|██████████| 1/1 [00:02<00:00,  2.67s/it]


query_embedding.shape: (14, 3072)


Extracting query embeddings...: 100%|██████████| 1/1 [00:02<00:00,  2.72s/it]


性能对比
FAISS索引检索: 2.762秒
直接计算检索: 2.878秒
加速比: 1.04x

注: 对于更大规模的文档集合，FAISS的优势会更明显


## 8. 加载ReRanker模型

In [ ]:
memory_monitor.print_memory("Before loading reranker")

# 初始化 Reranker
reranker = ImageReranker(
    model_name="monovlm",
    device=f"cuda:{DEVICE}",
    use_fast=True
)

memory_monitor.print_memory("After loading reranker")
print("✓ Reranker 加载完成")

[Before loading reranker] GPU Memory - Allocated: 8.21GB, Reserved: 8.87GB
Loading monovlm reranker on cuda:9...
Loading default monovlm model for language en
Default Model: lightonai/MonoQwen2-VL-v0.1
Loading MonoVLMRanker model lightonai/MonoQwen2-VL-v0.1 (this message can be suppressed by setting verbose=0)
bf16
Using dtype torch.bfloat16
Loading model lightonai/MonoQwen2-VL-v0.1, this might take a while...
Using device cuda:9.
Using dtype torch.bfloat16.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.83it/s]


VLM true token set to True
VLM false token set to False
✓ Reranker loaded successfully
[After loading reranker] GPU Memory - Allocated: 12.40GB, Reserved: 12.86GB
✓ Reranker 加载完成


## 7. 批量 Reranking

In [10]:
print("\n" + "="*80)
print("批量 Reranking")
print("="*80)

# 准备候选图片
all_candidate_images = []
for results in all_results:
    candidate_images = [images[r['page_index']] for r in results]
    all_candidate_images.append(candidate_images)

# 批量 reranking
all_rerank_results = reranker.rerank_batch(
    queries=queries,
    all_images_list=all_candidate_images,
    top_k=5
)

# 显示 reranking 结果
for i, (query, rerank_results) in enumerate(zip(queries, all_rerank_results)):
    print(f"\n查询 {i+1}: {query}")
    print("-"*80)
    print("Reranking Top-5:")
    for rr in rerank_results:
        original_result = all_results[i][rr['doc_id']]
        print(f"  {rr['rank']}. 第 {original_result['page_num']} 页")
        print(f"      Rerank分数: {rr['score']:.4f}, 检索分数: {original_result['score']:.4f}")

print("\n✓ Reranking 完成")


批量 Reranking

查询 1: 2022年员工总数是多少？
--------------------------------------------------------------------------------
Reranking Top-5:
  1. 第 101 页
      Rerank分数: 0.7500, 检索分数: 27.3640
  2. 第 44 页
      Rerank分数: 0.0549, 检索分数: 24.5208
  3. 第 90 页
      Rerank分数: 0.0198, 检索分数: 25.8386
  4. 第 91 页
      Rerank分数: 0.0145, 检索分数: 26.6214
  5. 第 100 页
      Rerank分数: 0.0132, 检索分数: 23.2531

查询 2: 以2021年作为基准年，2023年温室气体范围3排放量是否达到了减少的目标？
--------------------------------------------------------------------------------
Reranking Top-5:
  1. 第 18 页
      Rerank分数: 1.0000, 检索分数: 26.2757
  2. 第 104 页
      Rerank分数: 0.9961, 检索分数: 26.2978
  3. 第 100 页
      Rerank分数: 0.0618, 检索分数: 22.3862
  4. 第 21 页
      Rerank分数: 0.0067, 检索分数: 19.9749
  5. 第 16 页
      Rerank分数: 0.0057, 检索分数: 17.0577

查询 3: 截至二零二四年每收入单位的温室气体排放总量是多少？
--------------------------------------------------------------------------------
Reranking Top-5:
  1. 第 100 页
      Rerank分数: 1.0000, 检索分数: 24.2465
  2. 第 18 页
      Rerank分数: 1.0000, 检

In [11]:
# 提取每个查询的 top-5 图片
all_top5_images = []
for i, rerank_results in enumerate(all_rerank_results):
    # 获取 top-5 图片
    top5_images = [
        all_candidate_images[i][rr['doc_id']] 
        for rr in rerank_results[:5]
    ]
    all_top5_images.append(top5_images)
    
    print(f"\n查询 {i+1}: {queries[i]}")
    print(f"  Top-5 页面:")
    for rank, rr in enumerate(rerank_results[:5], 1):
        original = all_results[i][rr['doc_id']]
        print(f"    {rank}. 第 {original['page_num']} 页 (分数: {rr['score']:.4f})")


查询 1: 2022年员工总数是多少？
  Top-5 页面:
    1. 第 101 页 (分数: 0.7500)
    2. 第 44 页 (分数: 0.0549)
    3. 第 90 页 (分数: 0.0198)
    4. 第 91 页 (分数: 0.0145)
    5. 第 100 页 (分数: 0.0132)

查询 2: 以2021年作为基准年，2023年温室气体范围3排放量是否达到了减少的目标？
  Top-5 页面:
    1. 第 18 页 (分数: 1.0000)
    2. 第 104 页 (分数: 0.9961)
    3. 第 100 页 (分数: 0.0618)
    4. 第 21 页 (分数: 0.0067)
    5. 第 16 页 (分数: 0.0057)

查询 3: 截至二零二四年每收入单位的温室气体排放总量是多少？
  Top-5 页面:
    1. 第 100 页 (分数: 1.0000)
    2. 第 18 页 (分数: 1.0000)
    3. 第 22 页 (分数: 0.0074)
    4. 第 17 页 (分数: 0.0037)
    5. 第 21 页 (分数: 0.0024)


In [30]:
import importlib
import nvidia_rag_with_faiss  # 需要重新导入模块本身

# 使用 reload 函数重新加载整个模块
importlib.reload(nvidia_rag_with_faiss)

# 重新从模块中获取更新后的类/函数
# 实际上，在 Python 中，reload(module) 会更新 module.VQAModel
# 但为了确保您使用的变量指向新的定义，重新执行导入语句是一个好习惯
from nvidia_rag_with_faiss import VQAModel
# 初始化 VQA
vqa_model = VQAModel(
    model_name="doubao-seed-1-6-vision-250815"
)

✓ VQA Model initialized with API: doubao-seed-1-6-vision-250815


In [ ]:
# 批量问答（每个查询使用 top-5 图片）
answers = vqa_model.answer_batch_with_multiple_images(
    queries=queries,
    all_images_list=all_top5_images,
    max_tokens=512
)

# 显示结果
for i, (query, answer) in enumerate(zip(queries, answers)):
    print(f"\n{'='*80}")
    print(f"问题 {i+1}: {query}")
    print(f"{'='*80}")
    print(f"使用页面: Top-5 综合分析")
    for rank, rr in enumerate(all_rerank_results[i][:5], 1):
        original = all_results[i][rr['doc_id']]
        print(f"  {rank}. 第 {original['page_num']} 页")
    print(f"\n答案:\n{answer}")

VQA Processing (Multi-Image):   0%|          | 0/3 [00:00<?, ?it/s]

VQA Processing (Multi-Image): 100%|██████████| 3/3 [02:23<00:00, 47.78s/it]


问题 1: 2022年员工总数是多少？
使用页面: Top-5 综合分析
  1. 第 101 页
  2. 第 44 页
  3. 第 90 页
  4. 第 91 页
  5. 第 100 页

答案:
在第一张图片（雇佣绩效表格）中，“员工总数”指标下，截至2022年12月31日的员工总数为 **61,328**。

问题 2: 以2021年作为基准年，2023年温室气体范围3排放量是否达到了减少的目标？
使用页面: Top-5 综合分析
  1. 第 18 页
  2. 第 104 页
  3. 第 100 页
  4. 第 21 页
  5. 第 16 页

答案:
要判断2023年温室气体**范围3排放量**是否达到以2021年为基准年的减少目标，需结合**目标要求**和**实际排放数据**分析：


### 1. 明确减排目标  
根据《气候与自然相关目标》（附录），范围3的减排目标为：**以2021年为基准年，到2030年范围三绝对排放量减少30%**（即2030年排放量 ≤ 2021年排放量 × 70%）。  


### 2. 提取范围3排放量数据  
从《ESG关键绩效表》的“温室气体”板块，范围3排放量（吨二氧化碳当量）的年度数据为：  
- 2021年（基准年）：需结合趋势推断（2022年为2,917,512.5吨，2021年排放量与2022年接近，可近似参考）。  
- 2023年：**2,957,122.0吨**。  


### 3. 对比分析（2023年 vs 2021年）  
假设2021年范围3排放量约为2022年的水平（2,917,512.5吨），则：  
- 2023年排放量（2,957,122.0吨）**高于**2021年（或近似的2022年）排放量，**未实现“减少”**，反而有所增加。  


### 结论  
2023年温室气体范围3排放量**未达到**以2021年为基准年的减少目标（排放量较基准年增加，而非减少）。  

（注：目标为“2030年减少30%”，2023年处于目标周期内，但当前排放量趋势为**增长**，距离“减少”的阶段性目标（或长期目标的阶段性进展）存在差距。）

问题 3: 截至二零二四年每收入单位的温室气体排放总量是多少？
使用页面: Top-5 综合分析
  1. 第 100 页
  2. 

: 

## 总结

### FAISS索引的优势：
1. **速度提升**: 对于大规模文档（>1000页），检索速度可提升10-100倍
2. **内存优化**: 索引可以压缩存储，减少内存占用
3. **可扩展性**: 支持百万级甚至亿级向量检索

### 索引类型选择：
- **Flat**: <10K页面，精确搜索
- **IVFFlat**: 10K-1M页面，平衡速度和精度
- **IVFPQ**: >1M页面，最快速度，略微损失精度

### 多文档检索支持：
本实现已支持多文档检索功能，参考 M3DocRAG 的设计理念：
- ✅ 跨文档跨页面检索
- ✅ 每文档单页检索（保证文档多样性）
- ✅ 统一的多文档索引

详细示例请参考：
- **`Nvidia_FAISS_MultiDoc.ipynb`** - 多文档检索完整示例
- **`MULTI_DOC_RETRIEVAL_GUIDE.md`** - 多文档检索功能指南
- **`M3DOCRAG_MULTIPAGE_ANALYSIS.md`** - M3DocRAG 框架分析